# Part teòrica

**Variables:** Caselles consecutives amb direcció vertical o horitzontal de longitud > 1

**Domini:** Paraules del diccionari

**Restriccions:**
- Files i columnes han d'estar dins del taulell
- Files i columnes > 1 casella
- Interseccions de files i columnes tenen la mateixa lletra
- Si es troba una # acaba la fila o la columna
- Les paraules han d'estar escrites de dalt a baix i d'esquerra a dreta
- No es pot repetir una paraula
- Les paraules han de tenir una llargada menor o igual al màxim de m o n del taulell

**Tamany espai de solucions inicial:** D^v on *D* es el domini de paraules del diccionari i *v* les variables

**Estratègia de millora:** 


# Codi

# **Exercici 1**

### Carregar biblioteques

In [153]:
import numpy as np
import timeit

### Loading data

In [154]:
def loadCrossword(file):
    data = []
    with open(file, 'r') as file:
        for line in file.readlines():
            elements = line.strip().split()
            data.append(elements)
    return(np.array(data))

def loadDictionary(file):
    words = {}
    with open(file, 'r') as file:
        for word in file.readlines():   
            if len(word.strip()) not in words.keys():
                words[len(word.strip())] = [word.strip()]
            else:
                v = words[len(word.strip())]
                v.append(word.strip())

    return words


In [155]:
def cercaVariablesHoritzontal(taulell, variables):
    for n, i in enumerate(taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    variables.append([initial_pos, length, 0])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

### Cerca de variables
```variable = [[pos_inicial], len, v/h]; vertical = 1, horitzontal = 0 ```

In [156]:
def cercaVariablesVertical(taulell, variables):
    transposed_taulell = taulell.transpose()
    for n, i in enumerate(transposed_taulell):
        length = 0
        initial_pos = [n, 0]
        for m, j in enumerate(i):
            if j == '#':
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            elif m == len(i) - 1:
                length += 1
                if length > 1:
                    initial_pos.reverse()
                    variables.append([initial_pos, length, 1])
                length = 0
                initial_pos = [n, m + 1]
            else: length += 1

In [157]:
def cercaVariables(taulell, variables):
    cercaVariablesHoritzontal(taulell, variables)
    cercaVariablesVertical(taulell, variables)

In [158]:
def calculaPosicionsVariable(variable):
    variable_positions = []
    for x in range(variable[1]):
        i, j = 0, 0
        if variable[2] == 0: i = x
        else: j = x
        variable_positions.append([variable[0][0]+j, variable[0][1]+i])
    
    return variable_positions

### Letter of intersection

In [159]:
def intersectLetter(variable, word, assigned_variable, pos):
    if variable[2] == 0:
        return assigned_variable[1][pos[0] - assigned_variable[0][0][0]] == word[pos[1] - variable[0][1]]
    elif variable[2] == 1:
        return assigned_variable[1][pos[1] - assigned_variable[0][0][1]] == word[pos[0] - variable[0][0]]
    else: return False

### ActualitzarDominis

In [ ]:
#FC_dictionary = {"index_variable":[possibles paraules]}
def actualitzarDominis(variables, assigned, word, FC_dictionary):
    for i, variable in enumerate(1, variables):
       for intersection in calculaPosicionsVariable(variable):
            if intersection in calculaPosicionsVariable(assigned):
                if FC_dictionary[i].empty(): 
                    return False
                else:
                    for possible_word in FC_dictionary[i]:
                        if !intersectLetter(variable, possible_word, [assigned, word], intersection): 
                            FC_dictionary[i].remove(possible_word)

    return True

### Comprovació de restriccions

In [160]:
def isValid(word, variable, assigned_variables, taulell):

    for assigned_variable in assigned_variables:    
        if word == assigned_variable[1]:
            return False
        
    for var in assigned_variables:
        for intersection in calculaPosicionsVariable(variable):
            if intersection in calculaPosicionsVariable(var[0]) and not intersectLetter(variable, word, var, intersection): return False
    return True

### Backtracking

```
Funcio Backtracking(LVA,LVNA,R,D)
    Si (LVNA és buida) llavors Retornar(LVA) fSi
    Var=Cap(LVNA);
    Per a cada (valor del Domini(Var, D) que podem assignar a Var) fer
        Si (SatisfaRestriccions([Var valor],LVA,R)) llavors
            Res=Backtracking(Insertar([Var, valor],LVA),Cua(LVNA),R,D);
            Si (Res és una solució completa) llavors
                Retornar(Res);
            Fsi
        Fsi
    Fper
    Retornar(Falla)
FFuncio

```

In [161]:
def backtracking(assigned_variables, variables, taulell, dictionary):
    if not variables:
        return assigned_variables

    var = variables[0]
    for word in dictionary[var[1]]:
        if isValid(word, var, assigned_variables, taulell):
         )   assigned_variables.append([var, word])
            result = backtracking(assigned_variables, variables[1:], taulell, dictionary)

            if result:
                return result

            assigned_variables.pop()  

    return None

### BackForwardChecking

In [ ]:
def forwardChecking():
    

### Print taulell final

In [162]:
def printSolution(assigned_variables, taulell):
    for variable, word in assigned_variables:
        positions = calculaPosicionsVariable(variable)
        for i, j in positions:
            taulell[i][j] = word[i - variable[0][0] if variable[2] == 1 else j - variable[0][1]]

    for row in taulell:
        print(" ".join(row))

### Main

In [163]:

if __name__ == '__main__':
    
    taulell = loadCrossword('crossword_CB_v3.txt')
    dictionary = loadDictionary('diccionari_CB_v3.txt')
    
    assigned_variables = []
    variables = []
    cercaVariables(taulell, variables)
    
    res = backtracking(assigned_variables, variables, taulell, dictionary)

    print("Resultat: ", res)
    
    printSolution(assigned_variables, taulell)
        
    
    

Resultat:  [[[[0, 0], 6, 0], 'CANTAR'], [[[2, 2], 4, 0], 'CLAN'], [[[4, 1], 5, 0], 'PREMI'], [[[5, 0], 4, 0], 'PIAR'], [[[6, 0], 2, 0], 'ON'], [[[0, 0], 4, 1], 'CARA'], [[[5, 0], 2, 1], 'PO'], [[[4, 1], 3, 1], 'PIN'], [[[4, 2], 2, 1], 'RA'], [[[0, 3], 6, 1], 'TALLER'], [[[0, 5], 5, 1], 'RANCI']]
C A N T A R
A # # A # A
R # C L A N
A # # L # C
# P R E M I
P I A R # #
O N # # # #
